# JN0e · One set of numbers, many pictures

*On-ramp 5 of 8.*

Permits rose and fell across eight years. What's the *honest* way to show that — and does the choice of picture quietly change the story? **A chart is an argument.** Let's make the same numbers argue several ways.

### Running the cells

To run a cell, click it and press **Shift + Return**, or click the **run (▸) button** on the cell. The simplest way through any notebook here is to start at the top and run each cell in order, reading the output that appears beneath it.

Some of the computational cells may look complex right now — that's expected, and it's fine. **You don't need to understand every line yet;** the ideas become clear as you go. Run them, watch what they produce, and keep moving.

💡 Tip: the **Next** link opens the following notebook in a new tab. If Colab says you have too many sessions, just close the previous tab and continue.

<!-- NAV:auto-generated by scripts/build_nav.py — do not edit by hand -->

← Previous: [JN0d · What a pandas DataFrame is](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN0d_dataframe.ipynb)  |  Next: [JN0f · The tools we use](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN0f_tools.ipynb) →

## (run first) Colab setup

Fetches the data + shared modules from R2. **No-op if you already have the repo locally.** On Colab it recreates the minimal layout.

In [ ]:
# === COLAB BOOTSTRAP - fetch curriculum data + modules from R2 (NO-OP if the repo is local) ===
from pathlib import Path
import sys, urllib.request, urllib.parse, tarfile, subprocess

R2 = 'https://pub-2cee87f70da64080ab70ee0a34b55099.r2.dev/curriculum'
USE_CLEAN = False   # False: raw .xlsx path (JN1's messy-data lesson).  True (skip-ingest): permits_clean.*

_here = Path.cwd()
def _repo_ok(_here):
    """True only if a scripts/ tree exists AND housing_rules actually imports from it.
    A stale Colab extraction satisfies 'the directory exists' while being unusable, which
    previously skipped both the module refetch AND the data fetch. Anything that cannot
    import is treated as absent; under /content (a disposable Colab tree, never a real
    checkout) the broken copy is removed so the fetch below replaces it."""
    import importlib, shutil
    for _base in [_here] + list(_here.parents):
        if not (_base/'scripts'/'build_v2').exists():
            continue
        sys.path.insert(0, str(_base/'scripts'))
        try:
            for _m in [k for k in list(sys.modules)
                       if k.split('.')[0] in ('housing_rules', 's0_keys', 'cpra_dedup')]:
                del sys.modules[_m]
            importlib.invalidate_caches()
            import housing_rules  # noqa: F401  - the real test: does the package satisfy its own __init__?
            return True
        except Exception as _e:
            print(f'modules present but unusable ({type(_e).__name__}: {_e}); refetching')
            try: sys.path.remove(str(_base/'scripts'))
            except ValueError: pass
            # Remove the broken tree ONLY where it is a downloaded extraction, never a real
            # checkout: a genuine repo has .git beside scripts/. Without this removal the
            # fetch below is skipped (its own guard also only tests existence) and the stale
            # copy survives — which is precisely the bug this replaces.
            if not (_base/'.git').exists():
                shutil.rmtree(_base/'scripts', ignore_errors=True)
                print('removed the unusable scripts/ tree; it will be re-downloaded')
            return False
    return False

_have_repo = _repo_ok(_here)

def _get(url):
    # r2.dev sits behind Cloudflare, which 403s the default 'Python-urllib' User-Agent; send a browser UA.
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read()

if _have_repo:
    print('local repo detected - no fetch needed')
else:
    try:
        import pyarrow  # the parquet / USE_CLEAN path needs it; Colab has pandas, maybe not pyarrow
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'], check=True)
    def _fetch(url, dest):
        dest = Path(dest)
        if dest.exists():
            return                                   # cached: re-runs don't re-download
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(_get(url)); print('fetched', dest.name)
    # 1) shared modules -> ./scripts/...  (the config-cell repo-root walk then finds scripts/build_v2)
    if not (_here/'scripts'/'build_v2').exists():
        Path('modules.tgz').write_bytes(_get(f'{R2}/curriculum_modules.tar.gz'))
        _tar = tarfile.open('modules.tgz')
        try: _tar.extractall(_here, filter='data')      # py3.12+: safe extract, no deprecation warning
        except TypeError: _tar.extractall(_here)         # older python has no filter arg
        _tar.close(); Path('modules.tgz').unlink(missing_ok=True)   # tidy: drop the intermediate tarball
        print('extracted modules -> ./scripts/')
    # 2) data -> the SAME relative paths the notebooks use (raw .xlsx AND clean exports, both fetched)
    for rel in ['data/raw/cpra-downloads/BP_Annual Permit Report-2018-2022.xlsx',
                'data/raw/cpra-downloads/BP_Annual Permit Report-2023-2025.xlsx',
                'databases/hcd_apr_mirror_2026-06-17_fresh.db',
                'databases/hcd_apr_mirror.db',
                'data/processed/permits_clean.csv',
                'data/processed/permits_clean.parquet',
                'data/processed/permits_clean_README.md']:
        _fetch(f"{R2}/data/{urllib.parse.quote(rel.split('/')[-1])}", _here/rel)   # quote -> %20 for the spaced .xlsx names
    print('curriculum bundle ready (fetched from R2)')


In [ ]:
def md(t):
    from IPython.display import Markdown, display
    display(Markdown(t))

## Point the notebook at the data

Finds the repo root, the permit feed, and the project's shared code. The two knobs near the top are all a student changes to run another city.

In [ ]:
# === CONFIG — point this at YOUR city's permit data (this notebook is clonable) ===
from pathlib import Path
import sys, glob

# walk up to the repo root (where scripts/build_v2 lives) so the notebook runs from anywhere
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'scripts' / 'build_v2').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

# --- the two knobs a student changes for another city ---
PERMIT_GLOB   = str(REPO_ROOT / 'data/raw/cpra-downloads/BP_Annual Permit Report-*.xlsx')
HEADER_ROW    = 7        # 0-indexed: Berkeley's CPRA export puts the column names on row 8
EXPECTED_UNIQUE = 30764  # the known unique-permit total for YOUR feed (Berkeley = 30,764)

# import the REAL shared modules the pipeline uses (we demonstrate them, never reinvent)
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
sys.path.insert(0, str(REPO_ROOT / 'scripts' / 'build_v2'))
print('repo root :', REPO_ROOT)
print('feed files:', [Path(f).name for f in glob.glob(PERMIT_GLOB)])


In [ ]:
import pandas as pd, glob
def _load(p):
    # read one spreadsheet at its real header row, then tidy the column names
    d = pd.read_excel(p, dtype=str, header=HEADER_ROW); d.columns = [str(c).strip() for c in d.columns]; return d
df = pd.concat([_load(f) for f in glob.glob(PERMIT_GLOB)], ignore_index=True)   # stack every yearly file into one table
df = df[df['PermitNumber'].notna()].copy()        # drop rows that have no permit number
df['units_n'] = pd.to_numeric(df['NumberUnits'], errors='coerce')   # add a numeric units column (bad values -> NaN)
df['year'] = df['PermitNumber'].str.extract(r'^[A-Za-z]+(\d{4})')[0]   # add a year column pulled from the permit number
print(f'{len(df):,} permits loaded, columns ready')

## The numbers, bare: permits per year

Start with the plainest view — a table of counts.

In [ ]:
per_year = df['year'].value_counts().sort_index()   # count permits per year, ordered oldest -> newest
per_year

## Same numbers, as a bar chart

A **bar chart** turns the table into a comparison your eye reads instantly — which years were busy.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,3.2))            # one chart canvas
ax.bar(per_year.index, per_year.values, color='#1f77b4')   # one bar per year
ax.set_title('Building permits per year'); ax.set_xlabel('year'); ax.set_ylabel('permits')
plt.tight_layout(); plt.show()

## Bar versus line — categories versus trend

The **same data** as a **line chart** argues *trajectory* instead of *comparison*. Neither is wrong; they make different claims.

In [ ]:
fig, ax = plt.subplots(figsize=(8,3.2))
ax.plot(per_year.index, per_year.values, marker='o', color='#d62728')   # the same yearly counts, drawn as a line
ax.set_title('Building permits per year (as a trend)'); ax.set_xlabel('year'); ax.set_ylabel('permits')
plt.tight_layout(); plt.show()

## A histogram asks a *different* question

Not *how many per year* but *how big are they?* A **histogram** bins the unit counts to show the shape of the distribution — and Berkeley's is almost all tiny permits with a long thin tail of big buildings.

In [ ]:
fig, ax = plt.subplots(figsize=(8,3.2))
# bin the per-permit unit counts (capped, so the long tail doesn't hide the shape)
ax.hist(df['units_n'].dropna().clip(upper=40), bins=40, color='#2ca02c')
ax.set_title('How big is each permit? (units, capped at 40 to see the shape)')
ax.set_xlabel('units on the permit'); ax.set_ylabel('number of permits')
plt.tight_layout(); plt.show()

## Small multiples — comparison by juxtaposition

Two little charts side by side often beat one busy one. Here, *New* construction vs *Alteration* over time — the housing story and the renovation story, separated.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11,3.2), sharey=True)   # two side-by-side charts, shared y-axis
for ax, wt, col in [(a1,'New','#ff7f0e'), (a2,'Alteration','#9467bd')]:
    s = df[df['Work Type']==wt]['year'].value_counts().sort_index()   # this work type's permits per year
    ax.bar(s.index, s.values, color=col); ax.set_title(wt+' permits / year')
    ax.set_xlabel('year'); ax.tick_params(axis='x', rotation=45)
a1.set_ylabel('permits'); plt.tight_layout(); plt.show()

## The same data in 3D — a picture you can walk around

We can stack a third axis: **year × building-size bucket × count**. matplotlib's built-in 3D (no extra install) draws it; rotating the camera (two angles below) is the "2½-D" feel. Note the honest caution — 3D can *hide* bars behind bars as easily as it reveals shape.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3d projection)
import numpy as np
# bucket each permit by size, then cross-tabulate year x size-bucket into a grid of counts
bucket = pd.cut(df['units_n'].fillna(0), [-1,0,1,5,20,1e9], labels=['0','1','2-5','6-20','20+'])
piv = pd.crosstab(df['year'], bucket)
yrs, bks = list(piv.index), list(piv.columns)
xpos, ypos, dz = [], [], []
# flatten the grid into (x, y, height) triples, one per bar
for xi, y in enumerate(yrs):
    for yi, b in enumerate(bks):
        xpos.append(xi); ypos.append(yi); dz.append(float(piv.loc[y, b]))
cols = plt.cm.viridis(np.array(dz)/max(dz))        # colour each bar by its height
fig = plt.figure(figsize=(12,4.6))
for i, az in enumerate([35, 125]):                 # draw the same scene from two camera angles
    ax = fig.add_subplot(1, 2, i+1, projection='3d')
    ax.bar3d(xpos, ypos, [0]*len(dz), 0.7, 0.7, dz, color=cols, shade=True)
    ax.set_xticks(range(len(yrs))); ax.set_xticklabels(yrs, rotation=45, fontsize=7)
    ax.set_yticks(range(len(bks))); ax.set_yticklabels(bks, fontsize=7)
    ax.set_xlabel('year', fontsize=8); ax.set_ylabel('units/permit', fontsize=8); ax.set_zlabel('permits', fontsize=8)
    ax.view_init(elev=25, azim=az); ax.set_title(f'azimuth {az}°', fontsize=9)
fig.suptitle('The permit landscape in 3D — year × building size × count'); plt.tight_layout(); plt.show()

## The most ambitious picture: the city itself

Every chart so far is an *abstraction*. The most literal picture puts buildings back where they stand — extruded in 3D over the real map. One honest caveat: this map shows Berkeley's **~184 tracked development projects** (the big ones, each labelled with its status and unit count) — **not** the 30,764-permit feed or the 1,385-building spine the rest of the course analyzes. It's a different, curated layer. Here's a screenshot of those 184 projects (the live, draggable version is the cell after):

In [ ]:
from pathlib import Path
from IPython.display import Image, display
_asset = REPO_ROOT/'notebooks/curriculum/assets/berkeley_maplibre.jpg'
if not _asset.exists():                             # fetch the screenshot from R2 only if it isn't already here
    import urllib.request
    _asset.parent.mkdir(parents=True, exist_ok=True)
    _req = urllib.request.Request('https://raw.githubusercontent.com/blockXblock/berkeley-housing-analysis/main/notebooks/curriculum/assets/berkeley_maplibre.jpg', headers={'User-Agent':'Mozilla/5.0'})
    _asset.write_bytes(urllib.request.urlopen(_req, timeout=60).read())
display(Image(filename=str(_asset)))               # show the saved map screenshot

### …and live, if your browser allows it

The map below is the **real** open-source map — **MapLibre GL JS** (an open fork of Mapbox, *no API key, no bill*) on a keyless **OpenFreeMap** basemap, drawing those **~184 tracked development projects** as orange 3D boxes. Drag to pan, scroll to zoom, right-drag to tilt. It's the live counterpart of the pre-recorded flyby on berkeleybuild.com — and the open stack is the whole point: anyone can run this.

In [ ]:
from IPython.display import HTML
_MAP_B64 = 'PCFET0NUWVBFIGh0bWw+CjxodG1sIGxhbmc9ImVuIj4KPGhlYWQ+CiAgPG1ldGEgY2hhcnNldD0iVVRGLTgiPgogIDx0aXRsZT5CZXJrZWxleSBIb3VzaW5nIOKAlCBNYXBMaWJyZSBEZW1vPC90aXRsZT4KICA8bGluayBocmVmPSJodHRwczovL3VucGtnLmNvbS9tYXBsaWJyZS1nbEA0LjcuMS9kaXN0L21hcGxpYnJlLWdsLmNzcyIgcmVsPSJzdHlsZXNoZWV0Ij4KICA8c2NyaXB0IHNyYz0iaHR0cHM6Ly91bnBrZy5jb20vbWFwbGlicmUtZ2xANC43LjEvZGlzdC9tYXBsaWJyZS1nbC5qcyI+PC9zY3JpcHQ+CiAgPHNjcmlwdCBzcmM9Imh0dHBzOi8vdW5wa2cuY29tL0B0bWN3L3RvZ2VvanNvbkA1LjguMS9kaXN0L3RvZ2VvanNvbi51bWQuanMiPjwvc2NyaXB0PgogIDxzdHlsZT4KICAgIGh0bWwsIGJvZHksICNtYXAgewogICAgICB3aWR0aDogMTAwJTsKICAgICAgaGVpZ2h0OiAxMDAlOwogICAgICBtYXJnaW46IDA7CiAgICAgIHBhZGRpbmc6IDA7CiAgICAgIG92ZXJmbG93OiBoaWRkZW47CiAgICB9CiAgICAjaW5mbyB7CiAgICAgIHBvc2l0aW9uOiBhYnNvbHV0ZTsKICAgICAgdG9wOiAxMnB4OwogICAgICBsZWZ0OiAxMnB4OwogICAgICBiYWNrZ3JvdW5kOiByZ2JhKDAsIDAsIDAsIDAuNyk7CiAgICAgIGNvbG9yOiB3aGl0ZTsKICAgICAgcGFkZGluZzogMTJweDsKICAgICAgYm9yZGVyLXJhZGl1czogNnB4OwogICAgICBmb250LWZhbWlseTogc2Fucy1zZXJpZjsKICAgICAgZm9udC1zaXplOiAxM3B4OwogICAgICBtYXgtd2lkdGg6IDMyMHB4OwogICAgICB6LWluZGV4OiAxMDA7CiAgICB9CiAgPC9zdHlsZT4KPC9oZWFkPgo8Ym9keT4KICA8ZGl2IGlkPSJtYXAiPjwvZGl2PgogIDxkaXYgaWQ9ImluZm8iPgogICAgPHN0cm9uZz5CZXJrZWxleSBIb3VzaW5nIFBpcGVsaW5lIOKAlCBNYXBMaWJyZSBEZW1vPC9zdHJvbmc+PGJyPgogICAgPHNwYW4gaWQ9InN0YXR1cyI+TG9hZGluZyBtYXAuLi48L3NwYW4+CiAgPC9kaXY+CiAgPHNjcmlwdD4KICAgIGNvbnN0IHN0YXR1c0VsID0gZG9jdW1lbnQuZ2V0RWxlbWVudEJ5SWQoJ3N0YXR1cycpOwoKICAgIGNvbnN0IG1hcCA9IG5ldyBtYXBsaWJyZWdsLk1hcCh7CiAgICAgIGNvbnRhaW5lcjogJ21hcCcsCiAgICAgIHN0eWxlOiAnaHR0cHM6Ly90aWxlcy5vcGVuZnJlZW1hcC5vcmcvc3R5bGVzL3Bvc2l0cm9uJywKICAgICAgY2VudGVyOiBbLTEyMi4yNywgMzcuODddLAogICAgICB6b29tOiAxNCwKICAgICAgcGl0Y2g6IDUwLAogICAgICBiZWFyaW5nOiAwLAogICAgfSk7CgogICAgbWFwLm9uKCdsb2FkJywgYXN5bmMgZnVuY3Rpb24gKCkgewogICAgICBzdGF0dXNFbC50ZXh0Q29udGVudCA9ICdGZXRjaGluZyBnZW9tZXRyeS5rbWwuLi4nOwoKICAgICAgdHJ5IHsKICAgICAgICBjb25zdCByZXNwb25zZSA9IGF3YWl0IGZldGNoKCdodHRwczovL2JlcmtlbGV5YnVpbGQuY29tL2dlb21ldHJ5LmttbCcpOwogICAgICAgIGNvbnN0IGttbFRleHQgPSBhd2FpdCByZXNwb25zZS50ZXh0KCk7CiAgICAgICAgY29uc3Qga21sRG9jID0gbmV3IERPTVBhcnNlcigpLnBhcnNlRnJvbVN0cmluZyhrbWxUZXh0LCAnYXBwbGljYXRpb24veG1sJyk7CiAgICAgICAgY29uc3QgZ2VvanNvbiA9IHRvR2VvSlNPTi5rbWwoa21sRG9jKTsKCiAgICAgICAgLy8gdG9nZW9qc29uIGNvbnZlcnRzIEtNTCA8TXVsdGlHZW9tZXRyeT4gdG8gR2VvSlNPTiBHZW9tZXRyeUNvbGxlY3Rpb24uCiAgICAgICAgLy8gT3VyIGxhYmVsZWQgUGxhY2VtYXJrcyBhbGwgdXNlIE11bHRpR2VvbWV0cnk8UG9pbnQgKyBQb2x5Z29uPiBmb3IgbGFiZWwgYW5jaG9yaW5nLgogICAgICAgIC8vIEV4dHJhY3QganVzdCB0aGUgUG9seWdvbiBmcm9tIGVhY2ggR2VvbWV0cnlDb2xsZWN0aW9uIHNvIE1hcExpYnJlIGNhbiBleHRydWRlIGl0LgogICAgICAgIGxldCB1bndyYXBwZWQgPSAwOwogICAgICAgIGdlb2pzb24uZmVhdHVyZXMuZm9yRWFjaChmID0+IHsKICAgICAgICAgIGlmIChmLmdlb21ldHJ5ICYmIGYuZ2VvbWV0cnkudHlwZSA9PT0gJ0dlb21ldHJ5Q29sbGVjdGlvbicpIHsKICAgICAgICAgICAgY29uc3QgcG9seWdvbiA9IGYuZ2VvbWV0cnkuZ2VvbWV0cmllcy5maW5kKGcgPT4gZy50eXBlID09PSAnUG9seWdvbicpOwogICAgICAgICAgICBpZiAocG9seWdvbikgewogICAgICAgICAgICAgIGYuZ2VvbWV0cnkgPSBwb2x5Z29uOwogICAgICAgICAgICAgIHVud3JhcHBlZCsrOwogICAgICAgICAgICB9CiAgICAgICAgICB9CiAgICAgICAgfSk7CiAgICAgICAgY29uc29sZS5sb2coYFVud3JhcHBlZCAke3Vud3JhcHBlZH0gR2VvbWV0cnlDb2xsZWN0aW9ucyB0byBQb2x5Z29uc2ApOwogICAgICAgIApsZXQgZXh0cmFjdGVkID0gMCwgZGVmYXVsdGVkID0gMDsKICAgICAgICBnZW9qc29uLmZlYXR1cmVzLmZvckVhY2goZiA9PiB7CiAgICAgICAgICBpZiAoZi5nZW9tZXRyeSAmJiBmLmdlb21ldHJ5LnR5cGUgPT09ICdQb2x5Z29uJykgewogICAgICAgICAgICAvLyBUcnkgdG8gZXh0cmFjdCBhbHRpdHVkZSBmcm9tIHRoZSBmaXJzdCB2ZXJ0ZXgncyB6LWNvb3JkaW5hdGUKICAgICAgICAgICAgY29uc3QgY29vcmRzID0gZi5nZW9tZXRyeS5jb29yZGluYXRlc1swXTsKICAgICAgICAgICAgbGV0IGhlaWdodCA9IG51bGw7CiAgICAgICAgICAgIGlmIChjb29yZHMubGVuZ3RoICYmIGNvb3Jkc1swXS5sZW5ndGggPj0gMykgewogICAgICAgICAgICAgIGhlaWdodCA9IGNvb3Jkc1swXVsyXTsKICAgICAgICAgICAgfQogICAgICAgICAgICAvLyBGYWxsIGJhY2s6IHBhcnNlIGhlaWdodCBmcm9tIEtNTCBkZXNjcmlwdGlvbiB0ZXh0ICgiSGVpZ2h0OiAzNS4wbSIpCiAgICAgICAgICAgIGlmIChoZWlnaHQgPT09IG51bGwgJiYgZi5wcm9wZXJ0aWVzLmRlc2NyaXB0aW9uKSB7CiAgICAgICAgICAgICAgY29uc3QgbWF0Y2ggPSBmLnByb3BlcnRpZXMuZGVzY3JpcHRpb24ubWF0Y2goL0hlaWdodDpccyooW1xkLl0rKVxzKm0vaSk7CiAgICAgICAgICAgICAgaWYgKG1hdGNoKSB7CiAgICAgICAgICAgICAgICBoZWlnaHQgPSBwYXJzZUZsb2F0KG1hdGNoWzFdKTsKICAgICAgICAgICAgICB9CiAgICAgICAgICAgIH0KICAgICAgICAgICAgLy8gRmluYWwgZmFsbGJhY2s6IDEybSAoYSByZWFzb25hYmxlIGRlZmF1bHQgZm9yIHVua25vd24gYnVpbGRpbmcgaGVpZ2h0KQogICAgICAgICAgICBpZiAoaGVpZ2h0ID09PSBudWxsIHx8IGlzTmFOKGhlaWdodCkpIHsKICAgICAgICAgICAgICBoZWlnaHQgPSAxMjsKICAgICAgICAgICAgICBkZWZhdWx0ZWQrKzsKICAgICAgICAgICAgfSBlbHNlIHsKICAgICAgICAgICAgICBleHRyYWN0ZWQrKzsKICAgICAgICAgICAgfQogICAgICAgICAgICBmLnByb3BlcnRpZXMuaGVpZ2h0ID0gaGVpZ2h0OwogICAgICAgICAgfQogICAgICAgIH0pOwogICAgICAgIGNvbnNvbGUubG9nKGBIZWlnaHRzOiAke2V4dHJhY3RlZH0gZXh0cmFjdGVkIGZyb20gY29vcmRzL2Rlc2NyaXB0aW9uLCAke2RlZmF1bHRlZH0gZGVmYXVsdGVkIHRvIDEybWApOwoKICAgICAgICBtYXAuYWRkU291cmNlKCdob3VzaW5nJywgewogICAgICAgICAgdHlwZTogJ2dlb2pzb24nLAogICAgICAgICAgZGF0YTogZ2VvanNvbiwKICAgICAgICB9KTsKCiAgICAgICAgbWFwLmFkZExheWVyKHsKICAgICAgICAgIGlkOiAnaG91c2luZy0zZCcsCiAgICAgICAgICB0eXBlOiAnZmlsbC1leHRydXNpb24nLAogICAgICAgICAgc291cmNlOiAnaG91c2luZycsCiAgICAgICAgICBmaWx0ZXI6IFsnPT0nLCAnJHR5cGUnLCAnUG9seWdvbiddLAogICAgICAgICAgcGFpbnQ6IHsKICAgICAgICAgICAgJ2ZpbGwtZXh0cnVzaW9uLWNvbG9yJzogJyNmZjk4MDAnLAogICAgICAgICAgICAnZmlsbC1leHRydXNpb24taGVpZ2h0JzogWydnZXQnLCAnaGVpZ2h0J10sCiAgICAgICAgICAgICdmaWxsLWV4dHJ1c2lvbi1iYXNlJzogMCwKICAgICAgICAgICAgJ2ZpbGwtZXh0cnVzaW9uLW9wYWNpdHknOiAwLjg1LAogICAgICAgICAgfSwKICAgICAgICB9KTsKCiAgICAgICAgY29uc3QgcG9seWdvbkNvdW50ID0gZ2VvanNvbi5mZWF0dXJlcy5maWx0ZXIoCiAgICAgICAgICBmID0+IGYuZ2VvbWV0cnkgJiYgZi5nZW9tZXRyeS50eXBlID09PSAnUG9seWdvbicKICAgICAgICApLmxlbmd0aDsKICAgICAgICBzdGF0dXNFbC50ZXh0Q29udGVudCA9ICdMb2FkZWQgJyArIHBvbHlnb25Db3VudCArICcgYnVpbGRpbmdzLic7CiAgICAgIH0gY2F0Y2ggKGVycm9yKSB7CiAgICAgICAgc3RhdHVzRWwudGV4dENvbnRlbnQgPSAnRXJyb3I6ICcgKyBlcnJvci5tZXNzYWdlOwogICAgICAgIGNvbnNvbGUuZXJyb3IoZXJyb3IpOwogICAgICB9CiAgICB9KTsKCiAgICBtYXAuYWRkQ29udHJvbChuZXcgbWFwbGlicmVnbC5OYXZpZ2F0aW9uQ29udHJvbCgpKTsKICA8L3NjcmlwdD4KPC9ib2R5Pgo8L2h0bWw+Cg=='
# The real ~130-line map runs in its own sandboxed frame. It loads MapLibre from a CDN and fetches
# the building geometry from berkeleybuild.com — so it draws live where your browser allows that
# (Colab / Jupyter usually do); in a static export it stays a frame. Drag to pan, scroll to zoom,
# right-drag to tilt.
HTML(f'<iframe src="data:text/html;base64,{_MAP_B64}" width="100%" height="500" style="border:0;border-radius:8px"></iframe>')

**Next — JN0f:** we've used pandas and matplotlib (and now MapLibre) without formally naming the toolkit. Let's name every tool and what it's for.

<!-- NAV:auto-generated by scripts/build_nav.py — do not edit by hand -->

← Previous: [JN0d · What a pandas DataFrame is](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN0d_dataframe.ipynb)  |  Next: [JN0f · The tools we use](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN0f_tools.ipynb) →